In [ ]:
import torch
import os
import random
import numpy as np


if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", DEVICE)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/crop_disease_project"

for folder in [
    "dataset",
    "splits",
    "models",
    "results",
    "gradcam"
]:
    os.makedirs(
        os.path.join(PROJECT_DIR, folder),
        exist_ok=True
    )

print("Project:", PROJECT_DIR)

In [ ]:
import glob
import os
import zipfile

zip_files = glob.glob("/content/*.zip")

print("ZIP files:")

for path in zip_files:
    size_gb = os.path.getsize(path) / (1024**3)
    print(
        os.path.basename(path),
        f"({size_gb:.2f} GB)"
    )

if not zip_files:
    raise FileNotFoundError(
        "Dataset ZIP not found."
    )

ZIP_PATH = zip_files[0]

print("\nSelected:", ZIP_PATH)

print(
    "Valid ZIP:",
    zipfile.is_zipfile(ZIP_PATH)
)

In [ ]:
import shutil
import os

DRIVE_DATASET_DIR = os.path.join(
    PROJECT_DIR,
    "dataset"
)

DRIVE_ZIP = os.path.join(
    DRIVE_DATASET_DIR,
    os.path.basename(ZIP_PATH)
)

if not os.path.exists(DRIVE_ZIP):

    shutil.copy2(
        ZIP_PATH,
        DRIVE_ZIP
    )

    print("Dataset copied to Drive.")

else:

    print(
        "Dataset already exists in Drive."
    )

print(DRIVE_ZIP)


In [ ]:
import os
import shutil
import subprocess

EXTRACT_DIR = "/Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf/Plant_leave_diseases_dataset_with_augmentation"

# # Remove the incomplete extraction from the failed attempt
# if os.path.exists(EXTRACT_DIR):
#     shutil.rmtree(EXTRACT_DIR)

# os.makedirs(EXTRACT_DIR, exist_ok=True)

# print("Dataset ZIP:")
# print(ZIP_PATH)

# print("\nStarting extraction...")

# result = subprocess.run(
#     [
#         "unzip",
#         "-q",
#         ZIP_PATH,
#         "-d",
#         EXTRACT_DIR
#     ],
#     capture_output=True,
#     text=True
# )

# if result.returncode != 0:
#     print("Extraction failed.")
#     print(result.stderr)
#     raise RuntimeError("Dataset extraction failed.")

# print("Extraction completed successfully.")
# print("Location:", EXTRACT_DIR)

In [ ]:
from pathlib import Path

for item in Path(EXTRACT_DIR).iterdir():

    if item.is_dir():
        print("📁", item.name)

    else:
        print("📄", item.name)

In [ ]:
from pathlib import Path

EXTRACT_DIR = Path(
    "/Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf/Plant_leave_diseases_dataset_with_augmentation"
)

print("Exists:", EXTRACT_DIR.exists())
print("Is directory:", EXTRACT_DIR.is_dir())

print("\nContents:")
for item in EXTRACT_DIR.iterdir():
    print(item)

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".JPG", ".JPEG", ".PNG", ".BMP"
}

def find_class_root(base_dir):

    candidates = []

    for root, dirs, files in os.walk(base_dir):

        image_dirs = []

        for d in dirs:

            folder = Path(root) / d

            count = sum(
                1
                for f in folder.iterdir()
                if f.is_file()
                and f.suffix in IMAGE_EXTENSIONS
            )

            if count > 0:
                image_dirs.append(d)

        if len(image_dirs) >= 2:
            candidates.append(
                (root, len(image_dirs))
            )

    if not candidates:
        raise RuntimeError(
            "Could not find class folders."
        )

    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return candidates[0][0]

EXTRACT_DIR="/Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf/Plant_leave_diseases_dataset_with_augmentation"

DATASET_ROOT = find_class_root(
    EXTRACT_DIR
)

print(
    "Dataset root:",
    DATASET_ROOT
)

In [ ]:
classes = sorted([
    d.name
    for d in Path(DATASET_ROOT).iterdir()
    if d.is_dir()
])

NUM_CLASSES = len(classes)

print(
    "Number of classes:",
    NUM_CLASSES
)

for i, cls in enumerate(classes):

    print(
        f"{i}: {cls}"
    )

In [ ]:
class_counts = {}
classes = sorted(
    [
        folder.name
        for folder in Path(DATASET_ROOT).iterdir()
        if folder.is_dir()
    ]
)

NUM_CLASSES = len(classes)

for cls in classes:
    folder = Path(
        DATASET_ROOT
    ) / cls

    count = sum(
        1
        for f in folder.iterdir()
        if f.is_file()
        and f.suffix in IMAGE_EXTENSIONS
    )

    class_counts[cls] = count


TOTAL_IMAGES = sum(
    class_counts.values()
)

print(
    "Total images:",
    TOTAL_IMAGES
)

print(
    "Total classes:",
    NUM_CLASSES
)

print("\nImages per class:")

for cls, count in sorted(
    class_counts.items()
):

    print(
        f"{cls}: {count}"
    )

In [ ]:
import pandas as pd

class_to_idx = {
    cls: i
    for i, cls in enumerate(classes)
}

records = []

for cls in classes:

    folder = Path(
        DATASET_ROOT
    ) / cls

    for image_path in folder.iterdir():

        if (
            image_path.is_file()
            and image_path.suffix
            in IMAGE_EXTENSIONS
        ):

            records.append({
                "image_path": str(image_path),
                "class_name": cls,
                "label": class_to_idx[cls]
            })

df = pd.DataFrame(records)

print(
    "Total records:",
    len(df)
)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [26]:

# PROJECT_DIR = "/Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf"
# train_df.to_csv(
#     os.path.join(
#         PROJECT_DIR,
#         "splits",
#         "train.csv"
#     ),
#     index=False
# )

# val_df.to_csv(
#     os.path.join(
#         PROJECT_DIR,
#         "splits",
#         "val.csv"
#     ),
#     index=False
# )

# test_df.to_csv(
#     os.path.join(
#         PROJECT_DIR,
#         "splits",
#         "test.csv"
#     ),
#     index=False
# )

# print("Splits saved.")

import os
import shutil

# ============================================================
# PROJECT DIRECTORY
# ============================================================

PROJECT_DIR = (
    "/Users/shyamchauhan/Desktop/home/codes/"
    "crop_disease_project/cnn leaf"
)

# ============================================================
# DATASET DIRECTORY
# ============================================================

DRIVE_DATASET_DIR = os.path.join(
    PROJECT_DIR,
    "dataset"
)

# ============================================================
# CREATE SPLITS DIRECTORY
# ============================================================

SPLITS_DIR = os.path.join(
    PROJECT_DIR,
    "splits"
)

os.makedirs(
    SPLITS_DIR,
    exist_ok=True
)

# ============================================================
# SAVE TRAINING SPLIT
# ============================================================

train_df.to_csv(
    os.path.join(
        SPLITS_DIR,
        "train.csv"
    ),
    index=False
)

# ============================================================
# SAVE VALIDATION SPLIT
# ============================================================

val_df.to_csv(
    os.path.join(
        SPLITS_DIR,
        "val.csv"
    ),
    index=False
)

# ============================================================
# SAVE TEST SPLIT
# ============================================================

test_df.to_csv(
    os.path.join(
        SPLITS_DIR,
        "test.csv"
    ),
    index=False
)

# ============================================================
# VERIFY
# ============================================================

print("Splits saved successfully!")

print(
    "Train:",
    os.path.join(SPLITS_DIR, "train.csv")
)

print(
    "Validation:",
    os.path.join(SPLITS_DIR, "val.csv")
)

print(
    "Test:",
    os.path.join(SPLITS_DIR, "test.csv")
)

Splits saved successfully!
Train: /Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf/splits/train.csv
Validation: /Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf/splits/val.csv
Test: /Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf/splits/test.csv


In [27]:
import json

CLASS_JSON = os.path.join(
    PROJECT_DIR,
    "classes.json"
)

with open(
    CLASS_JSON,
    "w"
) as f:

    json.dump(
        class_to_idx,
        f,
        indent=2
    )

print(
    "Saved:",
    CLASS_JSON
)

Saved: /Users/shyamchauhan/Desktop/home/codes/crop_disease_project/cnn leaf/classes.json


In [28]:
from torchvision import transforms

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225
]

train_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        15
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

eval_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    )
])

In [29]:
from torch.utils.data import Dataset
from PIL import Image

class CropDiseaseDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe.reset_index(drop=True)
        )

        self.transform = transform


    def __len__(self):

        return len(self.dataframe)


    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        label = int(
            row["label"]
        )

        if self.transform:

            image = self.transform(
                image
            )

        return image, label

In [32]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

NUM_WORKERS = min(
    4,
    os.cpu_count() or 2
)

train_dataset = CropDiseaseDataset(
    train_df,
    train_transform
)

val_dataset = CropDiseaseDataset(
    val_df,
    eval_transform
)

test_dataset = CropDiseaseDataset(
    test_df,
    eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0
)

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     pin_memory=True,
#     persistent_workers=NUM_WORKERS > 0
# )

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

print("DataLoaders ready.")

DataLoaders ready.


In [33]:
images, labels = next(
    iter(train_loader)
)

print(
    "Images:",
    images.shape
)

print(
    "Labels:",
    labels.shape
)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])


In [34]:
import torch.nn as nn

from torchvision.models import (
    resnet50,
    ResNet50_Weights
)

model = resnet50(
    weights=ResNet50_Weights.DEFAULT
)

embedding_dim = model.fc.in_features

print(
    "Image embedding:",
    embedding_dim
)

model.fc = nn.Sequential(

    nn.Dropout(
        0.3
    ),

    nn.Linear(
        embedding_dim,
        NUM_CLASSES
    )
)

model = model.to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/shyamchauhan/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:07<00:00, 13.3MB/s]

Image embedding: 2048


In [35]:
train_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .values
)

class_weights = (
    len(train_df)
    /
    (
        NUM_CLASSES
        *
        train_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print("Weighted Cross Entropy ready.")

Weighted Cross Entropy ready.


In [36]:
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("AdamW ready.")

AdamW ready.


In [37]:
MAX_EPOCHS = 20

EARLY_STOPPING_PATIENCE = 3

MIN_DELTA = 0.001

print(
    "Maximum epochs:",
    MAX_EPOCHS
)

print(
    "Patience:",
    EARLY_STOPPING_PATIENCE
)

Maximum epochs: 20
Patience: 3


In [38]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS
)

In [39]:
USE_AMP = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    "Mixed precision:",
    USE_AMP
)

Mixed precision: False


In [40]:
MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models"
)

LATEST_CHECKPOINT = os.path.join(
    MODEL_DIR,
    "resnet50_latest.pth"
)

BEST_MODEL = os.path.join(
    MODEL_DIR,
    "best_resnet50.pth"
)

print(
    "Checkpoint system ready."
)

Checkpoint system ready.


In [42]:
import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    use_amp=True
):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )
        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp
        ):
            outputs = model(images)
            loss = criterion(
                outputs,
                labels
            )

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        preds = torch.argmax(
            outputs,
            dim=1
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    epoch_loss = (
        running_loss /
        len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return epoch_loss, accuracy, macro_f1

In [43]:
MAX_EPOCHS = 20
PATIENCE = 3
MIN_DELTA = 0.001

best_val_f1 = -np.inf
best_epoch = 0
patience_counter = 0

history = []

use_amp = (
    torch.cuda.is_available()
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("AMP enabled:", use_amp)

Maximum epochs: 20
Early stopping patience: 3
AMP enabled: False


In [44]:
def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    best_val_f1,
    history
):

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_f1": best_val_f1,
        "history": history
    }

    torch.save(
        checkpoint,
        path
    )


def load_checkpoint(
    path,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None
):

    checkpoint = torch.load(
        path,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

    if scheduler is not None:
        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

    if scaler is not None:
        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    return checkpoint

In [45]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)

        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "macro_f1": macro_f1
    }


print("✅ validate() function is ready")

✅ validate() function is ready


In [ ]:
# ============================================================
# RESNET-50 TRAINING
# ============================================================

import time
import copy
import numpy as np
import torch

# Training settings
MAX_EPOCHS = 30
PATIENCE = 5

best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

best_model_state = copy.deepcopy(model.state_dict())

print("=" * 70)
print("STARTING RESNET-50 TRAINING")
print("=" * 70)

for epoch in range(1, MAX_EPOCHS + 1):

    start_time = time.time()

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision
        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if DEVICE.type == "cuda":

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:

            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_metrics = validate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )

    val_loss = val_metrics["loss"]
    val_acc = val_metrics["accuracy"]
    val_precision = val_metrics["precision"]
    val_recall = val_metrics["recall"]
    val_f1 = val_metrics["macro_f1"]

    # --------------------------------------------------------
    # LEARNING RATE SCHEDULER
    # --------------------------------------------------------

    if scheduler is not None:

        # If using ReduceLROnPlateau
        if isinstance(
            scheduler,
            torch.optim.lr_scheduler.ReduceLROnPlateau
        ):
            scheduler.step(val_f1)

        else:
            scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)
    history["val_f1"].append(val_f1)

    elapsed = time.time() - start_time

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print()
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS}")
    print("-" * 70)

    print(
        f"Train Loss     : {train_loss:.4f}"
    )

    print(
        f"Train Accuracy : {train_acc:.4f}"
    )

    print(
        f"Val Loss       : {val_loss:.4f}"
    )

    print(
        f"Val Accuracy   : {val_acc:.4f}"
    )

    print(
        f"Val Precision  : {val_precision:.4f}"
    )

    print(
        f"Val Recall     : {val_recall:.4f}"
    )

    print(
        f"Val Macro-F1   : {val_f1:.4f}"
    )

    print(
        f"Learning Rate  : {current_lr:.2e}"
    )

    print(
        f"Time           : {elapsed:.1f}s"
    )

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        print(
            f"⭐ New best model! "
            f"Macro-F1 = {best_val_f1:.4f}"
        )

        # Save best model
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict()
                if optimizer is not None else None,
                "scheduler_state_dict": scheduler.state_dict()
                if scheduler is not None else None,
                "best_val_f1": best_val_f1,
                "class_names": class_names
            },
            BEST_MODEL_PATH
        )

        print(
            f"💾 Saved: {BEST_MODEL_PATH}"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if epochs_without_improvement >= PATIENCE:

        print()
        print(
            "🛑 Early stopping triggered."
        )

        break


# ------------------------------------------------------------
# RESTORE BEST MODEL
# ------------------------------------------------------------

model.load_state_dict(best_model_state)

print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Epoch     : {best_epoch}"
)

print(
    f"Best Val F1    : {best_val_f1:.4f}"
)

print(
    f"Total Epochs   : {len(history['train_loss'])}"
)


STARTING RESNET-50 TRAINING


KeyboardInterrupt: 

In [ ]:
# Fix missing class_names
try:
    class_names
except NameError:
    if "classes" in globals():
        class_names = classes
    elif hasattr(train_loader.dataset, "classes"):
        class_names = train_loader.dataset.classes
    elif hasattr(train_loader.dataset.dataset, "classes"):
        class_names = train_loader.dataset.dataset.classes
    else:
        raise RuntimeError(
            "Could not find class names. "
            "Please show me the code where train_loader was created."
        )

print("Number of classes:", len(class_names))
print("First classes:", class_names[:10])

In [ ]:
# ============================================================
# FIX — DEFINE BEST MODEL PATH
# ============================================================

import os

BEST_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "best_resnet50.pth"
)

os.makedirs(
    os.path.dirname(BEST_MODEL_PATH),
    exist_ok=True
)

print("Best model will be saved to:")
print(BEST_MODEL_PATH)

In [ ]:
# ============================================================
# SAVE CURRENT BEST MODEL
# ============================================================

import os
import torch

os.makedirs(os.path.dirname(BEST_MODEL_PATH), exist_ok=True)

torch.save(
    {
        "epoch": 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": (
            scheduler.state_dict()
            if scheduler is not None else None
        ),
        "best_val_f1": best_val_f1,
        "class_names": class_names,
    },
    BEST_MODEL_PATH
)

print("✅ Model checkpoint saved successfully.")
print("📁 Location:", BEST_MODEL_PATH)

In [ ]:
# ============================================================
# STEP 27 — RESUME RESNET-50 TRAINING FROM EPOCH 2
# ============================================================

import time
import copy
import torch

MAX_EPOCHS = 30
PATIENCE = 5

start_epoch = 2
epochs_without_improvement = 0

# Epoch 1 is already recorded in history
best_val_f1 = history["val_f1"][0]
best_epoch = 1

best_model_state = copy.deepcopy(model.state_dict())

print("=" * 70)
print("RESUMING RESNET-50 TRAINING")
print("=" * 70)
print(f"Starting from epoch: {start_epoch}")
print(f"Current best Val Macro-F1: {best_val_f1:.4f}")

for epoch in range(start_epoch, MAX_EPOCHS + 1):

    start_time = time.time()

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):

            outputs = model(images)
            loss = criterion(outputs, labels)

        if DEVICE.type == "cuda":

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:

            loss.backward()
            optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = validate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )

    val_loss = val_metrics["loss"]
    val_acc = val_metrics["accuracy"]
    val_precision = val_metrics["precision"]
    val_recall = val_metrics["recall"]
    val_f1 = val_metrics["macro_f1"]

    # ========================================================
    # SCHEDULER
    # ========================================================

    if scheduler is not None:

        if isinstance(
            scheduler,
            torch.optim.lr_scheduler.ReduceLROnPlateau
        ):
            scheduler.step(val_f1)
        else:
            scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)
    history["val_f1"].append(val_f1)

    elapsed = time.time() - start_time

    # ========================================================
    # RESULTS
    # ========================================================

    print()
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS}")
    print("-" * 70)

    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_acc:.4f}")
    print(f"Val Loss       : {val_loss:.4f}")
    print(f"Val Accuracy   : {val_acc:.4f}")
    print(f"Val Precision  : {val_precision:.4f}")
    print(f"Val Recall     : {val_recall:.4f}")
    print(f"Val Macro-F1   : {val_f1:.4f}")
    print(f"Learning Rate  : {current_lr:.2e}")
    print(f"Time           : {elapsed:.1f}s")

    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                    if scheduler is not None else None
                ),
                "best_val_f1": best_val_f1,
                "class_names": class_names
            },
            BEST_MODEL_PATH
        )

        print(
            f"⭐ New BEST model — Macro-F1: {best_val_f1:.4f}"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/{PATIENCE})"
        )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= PATIENCE:

        print()
        print("🛑 Early stopping triggered.")
        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

model.load_state_dict(best_model_state)

print()
print("=" * 70)
print("RESNET-50 TRAINING COMPLETE")
print("=" * 70)

print(f"Best Epoch  : {best_epoch}")
print(f"Best Val F1 : {best_val_f1:.4f}")
print(f"Epochs Run  : {len(history['train_loss'])}")
print(f"Model       : {BEST_MODEL_PATH}")

In [ ]:
# ============================================================
# FIX — DEFINE COMPUTATION DEVICE
# ============================================================

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU not detected — using CPU")

In [ ]:
# ============================================================
# STEP 26 — LOAD BEST RESNET-50 CHECKPOINT
# ============================================================

import os
import torch

print("Loading best ResNet-50 checkpoint...")

# Check that checkpoint exists
if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found:\n{BEST_MODEL_PATH}"
    )

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

best_f1 = checkpoint.get(
    "best_val_f1",
    None
)

print("✅ Best ResNet-50 model loaded")
print("📁 Path:", BEST_MODEL_PATH)

if best_f1 is not None:
    print(f"🏆 Best validation Macro-F1: {best_f1:.4f}")

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/crop_disease_project"

print("PROJECT:", PROJECT_DIR)
print("=" * 60)

for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{indent}  📄 {file}  ({size_mb:.2f} MB)")

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/crop_disease_project/models/best_resnet50.pth"

if os.path.isfile(MODEL_PATH):
    size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)
    print("✅ Model exists")
    print("📁", MODEL_PATH)
    print(f"📦 Size: {size_mb:.2f} MB")
else:
    print("❌ Model NOT found")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU is NOT enabled")

In [ ]:
import torch

device = torch.device("cpu")

print("Device:", device)
print("GPU available:", torch.cuda.is_available())

In [ ]:
import torch
import os

device = torch.device("cpu")

MODEL_PATH = "/content/drive/MyDrive/crop_disease_project/models/best_resnet50.pth"

print("Device:", device)
print("Model exists:", os.path.isfile(MODEL_PATH))
print("Model size:", round(os.path.getsize(MODEL_PATH) / (1024**2), 2), "MB")

In [ ]:
print("Loading best ResNet-50...")

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("✅ Best ResNet-50 loaded successfully.")

if "best_val_f1" in checkpoint:
    print(f"🏆 Best validation Macro-F1: {checkpoint['best_val_f1']:.4f}")

if "class_names" in checkpoint:
    class_names = checkpoint["class_names"]
    print("Classes:", class_names)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# ============================================================
# LOAD SAVED RESNET-50 CHECKPOINT
# ============================================================

MODEL_PATH = "/content/drive/MyDrive/crop_disease_project/models/best_resnet50.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Load checkpoint
checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

print("✅ Checkpoint loaded")

# Get class information
class_names = checkpoint["class_names"]
num_classes = len(class_names)

print("Number of classes:", num_classes)

# ============================================================
# RECREATE EXACT TRAINING ARCHITECTURE
# ============================================================

model = models.resnet50(weights=None)

# IMPORTANT:
# The original trained model used Dropout + Linear.
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.fc.in_features, num_classes)
)

# Load trained weights
model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Move model to available device
model = model.to(device)

# Evaluation mode
model.eval()

print("✅ Best ResNet-50 loaded successfully")
print("Device:", device)
print("Classes:", num_classes)

if "best_val_f1" in checkpoint:
    print(
        f"🏆 Best validation Macro-F1: "
        f"{checkpoint['best_val_f1']:.4f}"
    )

In [ ]:
# ============================================================
# VERIFY MODEL
# ============================================================

print("Model verification")
print("-" * 50)

print("Device:", device)
print("Number of classes:", len(class_names))
print("Model output classes:", model.fc[1].out_features)

print("\nFirst 10 classes:")
for i, name in enumerate(class_names[:10]):
    print(i, ":", name)

print("\n✅ Model architecture matches checkpoint")

In [ ]:
print("Checking test loader...")

print("Device:", device)
print("Number of classes:", len(class_names))
print("Test batches:", len(test_loader))

print("✅ Ready for test evaluation")

In [ ]:
# ============================================================
# CREATE TEST LOADER
# ============================================================

import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Paths
PROJECT_DIR = "/content/drive/MyDrive/crop_disease_project"
TEST_CSV = os.path.join(PROJECT_DIR, "splits", "test.csv")

print("Test CSV:", TEST_CSV)
print("Exists:", os.path.exists(TEST_CSV))

In [ ]:
# ============================================================
# LOAD TEST CSV
# ============================================================

test_df = pd.read_csv(TEST_CSV)

print("Test samples:", len(test_df))
print("\nColumns:")
print(test_df.columns.tolist())

print("\nFirst 5 rows:")
display(test_df.head())

In [ ]:
# ============================================================
# TEST DATASET
# ============================================================

class CropDiseaseTestDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# ============================================================
# TEST TRANSFORMS
# ============================================================

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_dataset = CropDiseaseTestDataset(
    test_df,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)

print("✅ Test dataset created")
print("Test images:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
print("Checking test loader...")

print("Device:", device)
print("Number of classes:", len(class_names))
print("Test batches:", len(test_loader))

print("✅ Ready for test evaluation")

In [ ]:
import os

print("Checking dataset locations...")

paths = [
    "/content/cnn_leaf_dataset",
    "/content/drive/MyDrive/crop_disease_project/dataset",
    "/content/drive/MyDrive/crop_disease_project",
]

for p in paths:
    print("\n", p)
    if os.path.exists(p):
        print("✅ EXISTS")
        print(os.listdir(p)[:20])
    else:
        print("❌ NOT FOUND")

In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/crop_disease_project/dataset/Plant_leaf_diseases_dataset_with_augmentation.zip"

print("ZIP exists:", os.path.isfile(ZIP_PATH))

if os.path.isfile(ZIP_PATH):
    print("Size:", os.path.getsize(ZIP_PATH) / (1024**2), "MB")

In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/crop_disease_project/dataset/Plant_leaf_diseases_dataset_with_augmentation.zip"

print("File exists:", os.path.isfile(ZIP_PATH))
print("File size:", os.path.getsize(ZIP_PATH) / (1024**2), "MB")

with open(ZIP_PATH, "rb") as f:
    header = f.read(20)

print("First 20 bytes:", header)

In [ ]:
import zipfile

ZIP_PATH = "/content/drive/MyDrive/crop_disease_project/dataset/Plant_leaf_diseases_dataset_with_augmentation.zip"

print("Checking ZIP integrity...")

print("Is valid ZIP:", zipfile.is_zipfile(ZIP_PATH))

if zipfile.is_zipfile(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        print("ZIP is valid")
        print("Number of files:", len(z.namelist()))
        print("First 20 files:")
        for x in z.namelist()[:20]:
            print(x)
else:
    print("❌ This file is NOT a valid ZIP file.")

In [ ]:
import os

BASE = "/content/drive/MyDrive/crop_disease_project"

print("Searching for image folders/files...\n")

for root, dirs, files in os.walk(BASE):
    # Don't scan huge unrelated folders
    if "models" in root or "results" in root or "gradcam" in root:
        continue

    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"))
    ]

    if image_files:
        print("IMAGE FOLDER:")
        print(root)
        print("Images:", len(image_files))
        print("Examples:", image_files[:3])
        print("-" * 60)

In [ ]:
import pandas as pd
import os

TEST_CSV = "/content/drive/MyDrive/crop_disease_project/splits/test.csv"

df = pd.read_csv(TEST_CSV)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nNumber of test images:", len(df))

In [ ]:
import os

print("\nChecking paths from test.csv...\n")

for col in df.columns:
    if "path" in col.lower() or "image" in col.lower() or "file" in col.lower():
        print("Possible image column:", col)
        print("Example:", df[col].iloc[0])

        p = str(df[col].iloc[0])
        print("Exists:", os.path.exists(p))

In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/crop_disease_project/dataset/Plant_leaf_diseases_dataset_with_augmentation.zip"

with open(ZIP_PATH, "rb") as f:
    data = f.read(100)

print("File:", ZIP_PATH)
print("Size:", os.path.getsize(ZIP_PATH) / (1024**2), "MB")
print("First 100 bytes:")
print(data)

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/crop_disease_project/dataset/Plant_leaf_diseases_dataset_with_augmentation.zip"

print("Checking ZIP in detail...")

try:
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        files = z.namelist()

        print("✅ ZIP opened successfully")
        print("Number of files:", len(files))

        print("\nFirst 20 entries:")
        for f in files[:20]:
            print(f)

        print("\nTesting archive integrity...")
        bad_file = z.testzip()

        if bad_file is None:
            print("✅ ZIP integrity is GOOD")
        else:
            print("❌ Corrupted file inside ZIP:", bad_file)

except Exception as e:
    print("❌ ZIP ERROR:")
    print(type(e).__name__, ":", e)

In [ ]:
import os
import pandas as pd

# Actual dataset root that your search found
REAL_DATASET_ROOT = "/content/cnn_leaf_dataset"

print("Checking real dataset root...")
print("Exists:", os.path.exists(REAL_DATASET_ROOT))

# Show what is actually inside
if os.path.exists(REAL_DATASET_ROOT):
    print(os.listdir(REAL_DATASET_ROOT)[:20])

In [ ]:
import os

print("Fixing test image paths...")

# Original paths from CSV
old_paths = df["image_path"].astype(str).tolist()

fixed_paths = []
missing = []

for old_path in old_paths:
    filename = os.path.basename(old_path)
    class_name = os.path.basename(os.path.dirname(old_path))

    # Expected real location
    new_path = os.path.join(
        REAL_DATASET_ROOT,
        "Plant_leaf_diseases_dataset_with_augmentation",
        class_name,
        filename
    )

    if os.path.exists(new_path):
        fixed_paths.append(new_path)
    else:
        fixed_paths.append(None)
        missing.append((class_name, filename, new_path))

print("Total test images:", len(old_paths))
print("Found:", len(fixed_paths) - len(missing))
print("Missing:", len(missing))

if missing:
    print("\nFirst 10 missing:")
    for x in missing[:10]:
        print(x)

In [ ]:
import os

ROOT = "/content/cnn_leaf_dataset"

print("Searching actual dataset structure...\n")

found_classes = {}

for root, dirs, files in os.walk(ROOT):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if image_files:
        class_name = os.path.basename(root)

        found_classes[class_name] = {
            "folder": root,
            "count": len(image_files),
            "examples": image_files[:3]
        }

print("Number of image folders found:", len(found_classes))

for i, (class_name, info) in enumerate(found_classes.items()):
    print(f"\n{i}: {class_name}")
    print("Folder:", info["folder"])
    print("Images:", info["count"])
    print("Examples:", info["examples"])

    if i >= 15:
        break